In [1]:
# =============================================================================
# Imports & Setup
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)


In [2]:
# =============================================================================
# LOAD DATA
# =============================================================================
def load_field_data(path='FieldData.txt'):
    df = pd.read_csv(path, sep='\t')
    df.columns = df.columns.str.strip()
    return df

df = load_field_data()
print(f"Loaded {len(df)} monthly field observations.")
df.head()


Loaded 360 monthly field observations.


,Time,C,R,Q,W,M1,M2,NetRevenue,Standard Deviation
0,Jan-96,1,74,57,37,1,5,-5.874630e+05,15207.84005
1,Feb-96,0,182,120,28,5,5,1.597538e+06,25628.12492
2,Mar-96,0,155,74,5,1,5,-2.243643e+05,23813.29448
3,Apr-96,1,176,149,42,2,2,-7.009770e+05,21584.93429
4,May-96,1,117,188,24,1,1,-8.913965e+05,24208.64863


In [3]:
# =============================================================================
# B-BICYCLE SIMULATOR (with rework loop) - from original script
# =============================================================================
class BBicycleSimulator:
    """DES: Assembly → Cleaning → Inspection → [Pass/Fail → Rework loop]"""
    def __init__(self):
        self.theta = {'assembly_mu':540,'assembly_sigma':25,'cleaning_mu':180,
                     'cleaning_sigma':20,'inspection_mu':60,'inspection_sigma':14,
                     'rework_mu':300,'rework_sigma':40,'fail_rate':0.12,'arrival_rate':6.25}

    def simulate(self, C, R, Q, W, M1, M2, n_days=30, n_reps=5):
        return np.mean([self._run(C,R,Q,W,M1,M2,n_days) for _ in range(n_reps)])

    def _run(self, C, R, Q, W, M1, M2, n_days):
        nr = -50000.0*(M1+M2); inv=50
        for day in range(n_days):
            for _ in range(np.random.poisson(self.theta['arrival_rate']*24)):
                if inv<=R: inv+=Q; nr-=100.0*Q
                if inv<=0: continue
                inv-=1
                ft = (max(10,np.random.normal(self.theta['assembly_mu'],self.theta['assembly_sigma']))
                    + max(5,np.random.normal(self.theta['cleaning_mu'],self.theta['cleaning_sigma']))/max(M1,1)
                    + max(3,np.random.normal(self.theta['inspection_mu'],self.theta['inspection_sigma']))/max(M2,1))
                rn=0
                while np.random.random()<self.theta['fail_rate'] and rn<3:
                    ft+=max(10,np.random.normal(self.theta['rework_mu'],self.theta['rework_sigma']))
                    ft+=max(3,np.random.normal(self.theta['inspection_mu'],self.theta['inspection_sigma']))/max(M2,1)
                    rn+=1; nr-=25.0
                fm=ft/60.0
                nr += (1000-5*fm) if C==0 else (500-2*fm)
                nr -= 1.5*(ft/3600.0)
        return nr

    def predict_batch(self, df, n_reps=3):
        return np.array([self.simulate(int(r['C']),int(r['R']),int(r['Q']),
            int(r['W']),int(r['M1']),int(r['M2']),n_reps=n_reps) for _,r in df.iterrows()])


In [4]:
# =============================================================================
# FEATURE ENGINEERING - from original script
# =============================================================================
def create_features(df):
    """Create rich feature set from decision variables."""
    X = df[['C','R','Q','W','M1','M2']].values.astype(float)
    feat = pd.DataFrame(X, columns=['C','R','Q','W','M1','M2'])

    # Interactions (based on domain knowledge from OFAT analysis)
    feat['C_x_W'] = feat['C'] * feat['W']        # Contract × MAXWIP
    feat['R_x_Q'] = feat['R'] * feat['Q']         # Reorder × Quantity
    feat['M1_x_M2'] = feat['M1'] * feat['M2']     # Machine interaction
    feat['M_total'] = feat['M1'] + feat['M2']      # Total machines
    feat['M_invest'] = 50000 * feat['M_total']      # Investment cost
    feat['W_inv'] = 1.0 / (feat['W'] + 1)          # Inverse WIP (nonlinear effect)
    feat['R_over_Q'] = feat['R'] / (feat['Q'] + 1)  # Reorder ratio
    feat['Q_x_W'] = feat['Q'] * feat['W']          # Quantity × WIP
    feat['C_x_M1'] = feat['C'] * feat['M1']
    feat['W_sq'] = feat['W'] ** 2                   # Quadratic W (strong nonlinear)
    feat['logW'] = np.log1p(feat['W'])
    feat['logR'] = np.log1p(feat['R'])
    feat['logQ'] = np.log1p(feat['Q'])

    return feat


In [5]:
# =============================================================================
# CALIBRATION: ENSEMBLE SURROGATE + DISCREPANCY - from original script
# =============================================================================
class EnsembleCalibrator:
    """
    KOH-inspired calibration using ensemble ML:

    y_field(x) = y_sim(x,θ) + δ(x) + ε

    Phase 1: Learn δ(x) using RF + GBR ensemble on training data
    Phase 2: Online recursive Bayesian correction on streaming data
    """
    def __init__(self):
        self.rf = None
        self.gbr = None
        self.bayesian_ridge = None
        self.scaler = StandardScaler()
        self.sim_scaler = StandardScaler()
        self.online_bias_history = [0.0]
        self.online_scale_history = [1.0]

    def calibrate_phase1(self, df_train, sim_preds, verbose=True):
        """Phase 1: Offline ensemble calibration."""
        if verbose:
            print("\n  Phase 1: Ensemble Surrogate Calibration (KOH Framework)")
            print("  " + "-" * 55)

        y_field = df_train['NetRevenue'].values
        y_sim = sim_preds

        # Feature matrix: decision variables + engineered features + sim predictions
        feat = create_features(df_train)
        feat['sim_pred'] = y_sim  # Include simulation output as a feature
        X = feat.values
        X_scaled = self.scaler.fit_transform(X)

        if verbose:
            rmse_sim = np.sqrt(mean_squared_error(y_field, y_sim))
            print(f"  Uncalibrated Sim Train RMSE: ${rmse_sim:,.0f} (R²={r2_score(y_field,y_sim):.4f})")

        # Ensemble: RF + GBR + BayesianRidge (stacking)
        if verbose: print("  Training Random Forest...")
        self.rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3,
                                       max_features='sqrt', random_state=42, n_jobs=-1)
        self.rf.fit(X_scaled, y_field)
        rf_pred = self.rf.predict(X_scaled)

        if verbose: print("  Training Gradient Boosting...")
        self.gbr = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                            min_samples_leaf=5, subsample=0.8, random_state=42)
        self.gbr.fit(X_scaled, y_field)
        gbr_pred = self.gbr.predict(X_scaled)

        # Stacking: combine RF + GBR predictions
        stack_X = np.column_stack([rf_pred, gbr_pred, y_sim])
        self.bayesian_ridge = BayesianRidge(max_iter=500)
        self.bayesian_ridge.fit(stack_X, y_field)

        # Final ensemble prediction (with uncertainty)
        final_pred = self.bayesian_ridge.predict(stack_X)

        # Cross-validation (on RF which is faster)
        cv = cross_val_score(RandomForestRegressor(n_estimators=200, max_depth=12,
                            min_samples_leaf=3, random_state=42),
                            X_scaled, y_field, cv=5, scoring='neg_root_mean_squared_error')
        cv_rmse = -cv.mean()

        train_rmse = np.sqrt(mean_squared_error(y_field, final_pred))
        train_r2 = r2_score(y_field, final_pred)

        if verbose:
            print(f"\n  Ensemble Train RMSE: ${train_rmse:,.0f} (R²={train_r2:.4f})")
            print(f"  5-Fold CV RMSE: ${cv_rmse:,.0f}")

            # Feature importance
            importances = self.rf.feature_importances_
            feat_names = list(feat.columns)
            top_idx = np.argsort(importances)[-6:][::-1]
            print(f"\n  Top features: {', '.join([f'{feat_names[i]}({importances[i]:.3f})' for i in top_idx])}")

        self.feat_columns = list(feat.columns)
        return {'train_pred': final_pred, 'cv_rmse': cv_rmse, 'train_rmse': train_rmse}

    def predict(self, df, sim_preds):
        """Predict using calibrated ensemble."""
        feat = create_features(df)
        feat['sim_pred'] = sim_preds
        X = self.scaler.transform(feat.values)

        rf_p = self.rf.predict(X)
        gbr_p = self.gbr.predict(X)
        stack = np.column_stack([rf_p, gbr_p, sim_preds])
        return self.bayesian_ridge.predict(stack)

    def predict_uncertainty(self, df, sim_preds):
        """Predict with Bayesian uncertainty."""
        feat = create_features(df)
        feat['sim_pred'] = sim_preds
        X = self.scaler.transform(feat.values)

        rf_p = self.rf.predict(X)
        gbr_p = self.gbr.predict(X)
        stack = np.column_stack([rf_p, gbr_p, sim_preds])
        mean, std = self.bayesian_ridge.predict(stack, return_std=True)

        # Add RF variance estimate
        tree_preds = np.array([t.predict(X) for t in self.rf.estimators_])
        rf_std = np.std(tree_preds, axis=0)
        total_std = np.sqrt(std**2 + rf_std**2)

        return mean, total_std

    def online_update(self, df_test, sim_preds_test, verbose=True):
        """Phase 2: Online Bayesian updating with streaming data."""
        if verbose:
            print("\n  Phase 2: Online Recursive Bayesian Updating")
            print("  " + "-" * 55)

        bias_mu = 0.0; bias_prec = 1e-14
        scale_mu = 1.0; scale_prec = 1e-12

        preds_p1 = []; preds_online = []
        self.online_bias_history = [0.0]; self.online_scale_history = [1.0]

        for i, (_, row) in enumerate(df_test.iterrows()):
            # Phase 1 prediction
            df_row = pd.DataFrame([row])
            sim_row = np.array([sim_preds_test[i]])
            p1 = self.predict(df_row, sim_row)[0]
            preds_p1.append(p1)

            # Apply online correction
            corrected = scale_mu * p1 + bias_mu
            preds_online.append(corrected)

            # Observe true value
            y_obs = row['NetRevenue']
            obs_noise_var = row['Standard Deviation']**2  # Use reported noise!
            obs_noise_var = max(obs_noise_var, 1e-6)  # noise floor to avoid instability
            noise_prec = 1.0 / obs_noise_var

            # Bayesian update for bias
            residual = y_obs - scale_mu * p1
            new_bp = bias_prec + noise_prec
            bias_mu = (bias_prec * bias_mu + noise_prec * residual) / new_bp
            bias_prec = new_bp

            # Bayesian update for scale
            if abs(p1) > 1e-4:
                obs_scale = (y_obs - bias_mu) / p1
                new_sp = scale_prec + noise_prec * p1**2
                scale_mu = (scale_prec * scale_mu + noise_prec * p1**2 * obs_scale) / new_sp
                scale_prec = new_sp
                scale_mu = np.clip(scale_mu, 0.3, 3.0)

            self.online_bias_history.append(bias_mu)
            self.online_scale_history.append(scale_mu)

            if verbose and i % 12 == 0:
                print(f"    Month {i:3d}: Pred=${corrected:>12,.0f} Obs=${y_obs:>12,.0f}"
                      f"  Bias=${bias_mu:>10,.0f} Scale={scale_mu:.3f}")

        return np.array(preds_p1), np.array(preds_online)


In [6]:
# =============================================================================
# DIGITAL TWIN PIPELINE - from original script
# =============================================================================
class DigitalTwinPipeline:
    def __init__(self, df):
        self.df = df
        self.sim = BBicycleSimulator()
        self.cal = EnsembleCalibrator()

    def run(self, train_frac=0.8, verbose=True):
        n = len(self.df); nt = int(n * train_frac)
        dtr = self.df.iloc[:nt].copy().reset_index(drop=True)
        dte = self.df.iloc[nt:].copy().reset_index(drop=True)
        otr = dtr['NetRevenue'].values; ote = dte['NetRevenue'].values

        if verbose:
            print("=" * 75)
            print("  DIGITAL TWIN: B-BICYCLE CALIBRATION WITH FIELD DATA")
            print("=" * 75)
            print(f"  Field Data: {len(self.df)} months (Jan-96 to Dec-25)")
            print(f"  Training: {len(dtr)} months | Testing: {len(dte)} months")
            print(f"  Model: DES with Rework Loop → Ensemble Surrogate Calibration")
            print(f"  Sync: Phase 1 (ML Calibration) + Phase 2 (Online Bayesian)")
            print("=" * 75)

        # Baseline simulation
        if verbose: print("\n  Running baseline simulation (uncalibrated)...")
        sim_tr = self.sim.predict_batch(dtr, n_reps=3)
        sim_te = self.sim.predict_batch(dte, n_reps=3)

        # Phase 1
        p1 = self.cal.calibrate_phase1(dtr, sim_tr, verbose=verbose)
        p1_te = self.cal.predict(dte, sim_te)
        p1_te_m, p1_te_s = self.cal.predict_uncertainty(dte, sim_te)

        if verbose:
            rmse = np.sqrt(mean_squared_error(ote, p1_te))
            print(f"\n  Phase 1 Test RMSE: ${rmse:,.0f} (R²={r2_score(ote,p1_te):.4f})")

        # Phase 2
        p1_stream, online_te = self.cal.online_update(dte, sim_te, verbose=verbose)

        self.results = {'dtr':dtr,'dte':dte,'otr':otr,'ote':ote,
                       'sim_tr':sim_tr,'sim_te':sim_te,
                       'p1_te':p1_te,'p1_te_s':p1_te_s,'online_te':online_te}

        # Final
        self.metrics = []
        methods = [("Baseline Simulation (No Calibration)", sim_te),
                  ("Phase 1: Ensemble Surrogate Calibration", p1_te),
                  ("Phase 2: Online Bayesian Updating", online_te)]

        if verbose:
            print("\n" + "=" * 75)
            print("  FINAL TEST RESULTS")
            print("=" * 75)
            print(f"\n  {'Method':<45} {'RMSE($)':>12} {'MAE($)':>12} {'MAPE':>8} {'R²':>8}")
            print(f"  {'-'*85}")

        for name, preds in methods:
            rmse = np.sqrt(mean_squared_error(ote, preds))
            mae = mean_absolute_error(ote, preds)
            nz = ote != 0; mape = np.mean(np.abs((preds[nz]-ote[nz])/ote[nz]))*100
            r2 = r2_score(ote, preds)
            self.metrics.append({'name':name,'rmse':rmse,'mae':mae,'mape':mape,'r2':r2})
            if verbose: print(f"  {name:<45} {rmse:>12,.0f} {mae:>12,.0f} {mape:>7.1f}% {r2:>8.4f}")

        if verbose:
            base = self.metrics[0]['rmse']
            for m in self.metrics[1:]:
                imp = (1 - m['rmse']/base) * 100
                print(f"\n  → {m['name'].split(':')[1].strip()}: {imp:.1f}% RMSE improvement over baseline")

        return self.results


In [7]:
pipeline = DigitalTwinPipeline(df)
results = pipeline.run(train_frac=0.8, verbose=True)

# ==========================================
# SAVE SIMULATION + PREDICTIONS TO EXCEL
# ==========================================

r = pipeline.results

# ---- Training data with simulation ----
train_export = r["dtr"].copy()
train_export["Simulated_NetRevenue"] = r["sim_tr"]

# ---- Test data with simulation + predictions ----
test_export = r["dte"].copy()
test_export["Simulated_NetRevenue"] = r["sim_te"]
test_export["Phase1_Prediction"] = r["p1_te"]
test_export["Phase2_Online_Prediction"] = r["online_te"]

# ---- Save to Excel ----
with pd.ExcelWriter("DigitalTwin_Output.xlsx") as writer:
    train_export.to_excel(writer, sheet_name="Training_Data", index=False)
    test_export.to_excel(writer, sheet_name="Test_Data", index=False)

print("Excel file saved as: DigitalTwin_Output.xlsx")

  DIGITAL TWIN: B-BICYCLE CALIBRATION WITH FIELD DATA
  Field Data: 360 months (Jan-96 to Dec-25)
  Training: 288 months | Testing: 72 months
  Model: DES with Rework Loop → Ensemble Surrogate Calibration
  Sync: Phase 1 (ML Calibration) + Phase 2 (Online Bayesian)

  Running baseline simulation (uncalibrated)...

  Phase 1: Ensemble Surrogate Calibration (KOH Framework)
  -------------------------------------------------------
  Uncalibrated Sim Train RMSE: $2,253,782 (R²=-5.6995)
  Training Random Forest...
  Training Gradient Boosting...

  Ensemble Train RMSE: $20,118 (R²=0.9995)
  5-Fold CV RMSE: $266,088

  Top features: M1_x_M2(0.185), M1(0.152), sim_pred(0.107), M_invest(0.078), M_total(0.065), C_x_M1(0.060)

  Phase 1 Test RMSE: $166,845 (R²=0.9568)

  Phase 2: Online Recursive Bayesian Updating
  -------------------------------------------------------
    Month   0: Pred=$     203,902 Obs=$     222,268  Bias=$    18,366 Scale=1.000
    Month  12: Pred=$   1,538,409 Obs=$   1,

In [8]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom = np.maximum(denom, eps)
    return np.mean(np.abs(y_pred - y_true) / denom) * 100.0

# Pull test-set observed and predictions from pipeline.results
r = pipeline.results
y = r["ote"]
preds = {
    "Baseline Simulation": r["sim_te"],
    "Phase 1 Calibrated": r["p1_te"],
    "Phase 2 Online": r["online_te"],
}

for name, p in preds.items():
    rmse = np.sqrt(mean_squared_error(y, p))
    mae = mean_absolute_error(y, p)
    r2 = r2_score(y, p)
    s = smape(y, p)
    print(f"{name:20s}  RMSE=${rmse:,.0f}  MAE=${mae:,.0f}  R²={r2:.4f}  sMAPE={s:.1f}%")


Baseline Simulation   RMSE=$2,275,417  MAE=$1,949,218  R²=-7.0414  sMAPE=129.0%
Phase 1 Calibrated    RMSE=$166,845  MAE=$132,515  R²=0.9568  sMAPE=41.0%
Phase 2 Online        RMSE=$173,732  MAE=$136,767  R²=0.9531  sMAPE=42.3%


In [ ]:
r = pipeline.results
ote = r["ote"]

plt.figure(figsize=(12,5))
plt.plot(ote/1e6, "r-o", ms=3, lw=1.5, label="Observed")
plt.plot(r["sim_te"]/1e6, "g--", lw=1.2, label="Baseline Simulation")
plt.plot(r["p1_te"]/1e6, "b-", lw=1.5, label="Phase 1 Calibrated")
plt.plot(r["online_te"]/1e6, "k-", lw=1.8, label="Phase 2 Online")
plt.axhline(0, color="gray", ls=":", lw=1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylabel("Net Revenue ($M)")
plt.xlabel("Test Month Index")
plt.title("Observed vs Predicted Net Revenue (Test Period)")
plt.show()


In [ ]:
pipeline = DigitalTwinPipeline(df)
results = pipeline.run(train_frac=0.8, verbose=True)


In [ ]:
r = pipeline.results
ote = r["ote"]          # observed test revenue
sim = r["sim_te"]       # baseline simulation
p1  = r["p1_te"]        # Phase 1 calibrated predictions
on  = r["online_te"]    # Phase 2 online predictions
p1_s = r.get("p1_te_s", None)   # may be None in some versions

In [ ]:
pred_sets = [
    ("Baseline Simulation", sim),
    ("Phase 1 Calibrated", p1),
    ("Phase 2 Online", on),
]

plt.figure(figsize=(16,5))

for i, (title, pred) in enumerate(pred_sets, start=1):
    ax = plt.subplot(1, 3, i)
    ax.scatter(ote, pred, alpha=0.6)
    
    # 45-degree reference
    vmin = min(ote.min(), pred.min())
    vmax = max(ote.max(), pred.max())
    ax.plot([vmin, vmax], [vmin, vmax], "--")
    
    rmse = np.sqrt(mean_squared_error(ote, pred))
    r2 = r2_score(ote, pred)
    
    ax.set_title(f"{title}\nRMSE=${rmse:,.0f}, R²={r2:.3f}")
    ax.set_xlabel("Observed NetRevenue")
    ax.set_ylabel("Predicted NetRevenue")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if p1_s is None:
    print("No uncertainty vector p1_te_s found in pipeline.results.")
else:
    t = np.arange(len(ote))
    lower = p1 - 1.96 * p1_s
    upper = p1 + 1.96 * p1_s

    plt.figure(figsize=(14,5))
    plt.fill_between(t, lower, upper, alpha=0.2, label="Phase 1 95% CI")
    plt.plot(t, ote, marker="o", markersize=3, linewidth=1.8, label="Observed")
    plt.plot(t, p1, linewidth=1.6, label="Phase 1 Mean Prediction")

    plt.axhline(0, linestyle=":", linewidth=1)
    plt.title("Phase 1 Predictions with 95% Confidence Interval (Test Period)")
    plt.xlabel("Test Month Index")
    plt.ylabel("Net Revenue ($)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
errs = [
    ("Baseline Error (Pred - Obs)", sim - ote),
    ("Phase 1 Error (Pred - Obs)", p1 - ote),
    ("Phase 2 Error (Pred - Obs)", on - ote),
]

plt.figure(figsize=(16,4.8))

for i, (title, e) in enumerate(errs, start=1):
    ax = plt.subplot(1, 3, i)
    ax.hist(e, bins=18, alpha=0.75, edgecolor="white")
    ax.axvline(0, linestyle="--", linewidth=1.5)
    ax.axvline(np.mean(e), linestyle="-", linewidth=1.5, label=f"Mean={np.mean(e):,.0f}")
    ax.set_title(title)
    ax.set_xlabel("Error ($)")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
if not hasattr(pipeline, "cal"):
    print("pipeline.cal not found (your notebook version may not store calibrator).")
elif (not hasattr(pipeline.cal, "online_bias_history")) or (not hasattr(pipeline.cal, "online_scale_history")):
    print("Online histories not found. Make sure Phase 2 ran and stored bias/scale history.")
else:
    bias_hist = np.array(pipeline.cal.online_bias_history)
    scale_hist = np.array(pipeline.cal.online_scale_history)
    t = np.arange(len(bias_hist))

    plt.figure(figsize=(12,6))

    ax1 = plt.subplot(2,1,1)
    ax1.plot(t, bias_hist)
    ax1.axhline(0, linestyle=":", linewidth=1)
    ax1.set_title("Online Bias History")
    ax1.set_ylabel("Bias ($)")
    ax1.grid(True, alpha=0.3)

    ax2 = plt.subplot(2,1,2)
    ax2.plot(t, scale_hist)
    ax2.axhline(1.0, linestyle=":", linewidth=1)
    ax2.set_title("Online Scale History")
    ax2.set_xlabel("Test Month Index")
    ax2.set_ylabel("Scale")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
names = ["Baseline", "Phase 1", "Phase 2"]
pred_list = [sim, p1, on]

rmses = [np.sqrt(mean_squared_error(ote, p)) for p in pred_list]
r2s   = [r2_score(ote, p) for p in pred_list]

plt.figure(figsize=(13,5))

ax1 = plt.subplot(1,2,1)
ax1.bar(names, rmses)
ax1.set_title("RMSE (Lower is better)")
ax1.set_ylabel("RMSE ($)")
ax1.grid(True, alpha=0.3, axis="y")

ax2 = plt.subplot(1,2,2)
ax2.bar(names, r2s)
ax2.set_title("R² (Higher is better)")
ax2.set_ylabel("R²")
ax2.axhline(0, linestyle=":", linewidth=1)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
t = np.arange(len(ote))
cum_baseline = np.cumsum(np.abs(sim - ote))
cum_p1       = np.cumsum(np.abs(p1  - ote))
cum_p2       = np.cumsum(np.abs(on  - ote))

plt.figure(figsize=(12,5))
plt.plot(t, cum_baseline, linestyle="--", linewidth=1.8, label="Baseline cumulative |error|")
plt.plot(t, cum_p1, linewidth=1.8, label="Phase 1 cumulative |error|")
plt.plot(t, cum_p2, linewidth=2.2, label="Phase 2 cumulative |error|")

plt.title("Cumulative Absolute Error Over Test Period")
plt.xlabel("Test Month Index")
plt.ylabel("Cumulative |Error| ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.savefig("fig_name.png", dpi=200, bbox_inches="tight")